In [1]:
import pandas as pd
from tqdm import tqdm
import numpy as np
import time

from __future__ import annotations
from yandex_cloud_ml_sdk import YCloudML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [2]:
X_test = pd.read_csv('X_test_balanced.csv',index_col=0)

In [3]:
X_test.head(10)

,purpose
393593,Перечисление средств по соглашению №2140 о предоставлении гранта победителю конкурса на предоставление грантов Раиса РТ на развитие гражданского общества от 26.11.2024г. НДС не облагается.
429170,ЗА 27/02/2025;СУМ:1500.00;КОМ:0.00;СЛЮСАРЕНКО ВИКТОРИЯ АНАТОЛЬЕВНА;Попечительский взнос котёнок Тужур
379020,"Перевод с карты *6947, Благотворительное пожертвование на уставную деятельность. НДС не облагается"
570841,"ОПЛАТА ПО ДОГОВОРУ НХТК.8190 (ДС НХТК.8190-2), СЧЕТ №21 ОТ 02.06.2025 (КВОТИРУЕМЫЕ РАБОЧИЕ МЕСТА). СУММА 80143-68 РУБ. БЕЗ НАЛОГА (НДС)"
485026,БЛАГОТВОРИТЕЛЬНАЯ ПОМОЩЬ ПО СОГЛАШЕНИЮ №БС2025-005 ОТ 16.01.2025 СУММА 310750-00 БЕЗ НАЛОГА (НДС)
591200,Выплата процентов по депозитной сделке № UNV-0000006342917 от 22.08.2025 за период с 23.08.2025 по 25.08.2025 согласно Правилам Банковского обслуживания. НДС не предусмотрен.
396289,"Перевод с карты *3120, Благотворительное пожертвование на уставную деятельность. НДС не облагается"
402359,Благотворительное пожертвование на уставную деятельность. НДС не облагается
147304,"Оплата по договору о предоставлении целевого гранта в рамках Благотворительной программы - акции ""Добрый новогодний подарок"" (2023-2024) № 24-1-000496 от 15.04.2024. 1 транш. НДС не облагается."
171375,Благотворительное пожертвование по договору 111-77/2024 от 29.02.2024НДС не облагается


In [5]:
y_pred_ygpt_ft = pd.DataFrame(columns=["universal_category"])

sdk = YCloudML(
        folder_id="YOUR_FOLDER_ID",
        auth="YOUR_YANDEX_CLOUD_TOKEN",
    )

model = sdk.models.text_classifiers(
        "cls://YOUR_FOLDER_ID/yandexgpt-lite/latest@tamros8p69qq7ribmm06k"
    )

for index__, text in tqdm(X_test['purpose'].items(), total=len(X_test)):
    attempts = 0
    max_attempts = 3
    while attempts < max_attempts:
        try:
            result = model.run(str(text))  # приводим текст к строке
            #print(result, "\n")
            best_label = max(result, key=lambda x: x.confidence).label
            y_pred_ygpt_ft.loc[index__, "universal_category"] = best_label
            break
        except Exception as e:
            attempts += 1
            print(f"Ошибка на index {index__}, попытка {attempts}/{max_attempts}: {e}")
            time.sleep(2)
    else:
        # если все попытки неудачные, присваиваем NaN
        y_pred_ygpt_ft.loc[index__, "universal_category"] = np.nan


100%|██████████| 900/900 [31:09<00:00,  2.08s/it]


In [6]:
y_pred_ygpt_ft.to_csv("y_pred_ygpt_ft.csv", index=True, header=["universal_category"])